<a href="https://colab.research.google.com/github/shin-ing-bot/data_playground/blob/main/data_join.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# 마스터 데이터베이스 구축


# Haybit 데이터 전처리

User_Profile.csv와 Event_Log.csv 전처리용 노트북

In [ ]:
import pandas as pd
import numpy as np

UP_PATH = "user_profile_clean.csv"
EL_PATH = "event_log_clean.csv"

up = pd.read_csv(UP_PATH, encoding="utf-8-sig")
el = pd.read_csv(EL_PATH, encoding="utf-8-sig")

print("user_profile_clean:", up.shape)
print("event_log_clean:", el.shape)

user_profile_clean: (12500, 6)
event_log_clean: (1757262, 7)


In [ ]:
# 기본 확인
display(up.head())
display(el.head())

print(up.dtypes)
print(el.dtypes)

print("\n[User_Profile 결측치]")
print(up.isnull().sum())

print("\n[Event_Log 결측치]")
print(el.isnull().sum())


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형,is_type_null,is_outage_period
0,U0000001,2025-01-25 07:25:45+09:00,앱실행,2858201769,NaN,False,False
1,U0000001,2025-01-25 07:26:15+09:00,온보딩_완료,2858201769,NaN,False,False
2,U0000001,2025-01-25 07:26:55+09:00,챌린지_탐색,2858201769,NaN,False,False
3,U0000001,2025-01-25 07:27:55+09:00,챌린지참여,2858201769,NaN,False,False
4,U0000001,2025-01-25 20:30:00+09:00,알림수신,NaN,광고성,False,False


User_ID        object
가입일자           object
가입경로           object
기기             object
알림수신동의여부       object
알림수신동의_변경일자    object
dtype: object
User_ID             object
Event_Time          object
Event_Type          object
Session_ID          object
알림_유형               object
is_type_null          bool
is_outage_period      bool
dtype: object

[User_Profile 결측치]
User_ID            0
가입일자               0
가입경로               0
기기                 0
알림수신동의여부           0
알림수신동의_변경일자    10524
dtype: int64

[Event_Log 결측치]
User_ID                   0
Event_Time                0
Event_Type            26456
Session_ID           241502
알림_유형               1538380
is_type_null              0
is_outage_period          0
dtype: int64


In [ ]:
# User_ID 형식 통일
up["User_ID"] = up["User_ID"].astype(str).str.strip()
el["User_ID"] = el["User_ID"].astype(str).str.strip()

In [ ]:
import pandas as pd
import numpy as np

# 날짜 변환
up["가입일자"] = pd.to_datetime(up["가입일자"])

up["알림수신동의_변경일자"] = pd.to_datetime(
    up["알림수신동의_변경일자"],
    errors="coerce"
)

el["Event_Time"] = pd.to_datetime(
    el["Event_Time"],
    format="mixed",
    errors="coerce"
)
el["Event_Time"] = el["Event_Time"].dt.tz_convert("Asia/Seoul")

In [ ]:
# 분석 기간 필터 (2025-01 ~ 2025-06)
el = el[
    (el["Event_Time"] >= "2025-01-01") &
    (el["Event_Time"] < "2025-07-01")
].copy()

print(el.shape)


(1757262, 7)


In [ ]:
# 결측치 처리

up["가입경로"] = up["가입경로"].fillna("미확인")
up["기기"] = up["기기"].fillna("미확인")

up["알림수신동의여부"] = (
    up["알림수신동의여부"]
    .astype(str)
    .replace("nan", "알 수 없음")
    .astype("category")
)

# Event_Type 결측 플래그
el["is_type_null"] = el["Event_Type"].isnull()

# 분석용 클린 데이터
el_clean = el.dropna(subset=["Event_Type"]).copy()

print("Event_Type 결측 제거 후:", el_clean.shape)


Event_Type 결측 제거 후: (1730806, 7)


In [ ]:
# Session_ID 결측 점검

notif_mask = el["Event_Type"].isin(["알림수신", "알림오픈"])

abnormal_null = el[
    (~notif_mask) &
    (el["Session_ID"].isnull())
]

print("비정상 Session_ID 결측:", len(abnormal_null))

alarm_type_null = el[
    notif_mask &
    (el["알림_유형"].isnull())
]

print("알림 이벤트 중 유형 결측:", len(alarm_type_null))


비정상 Session_ID 결측: 25959
알림 이벤트 중 유형 결측: 0


In [ ]:
# 3월 10~14일 수집 장애 플래그

outage = (
    (el["Event_Time"].dt.date >= pd.Timestamp("2025-03-10").date()) &
    (el["Event_Time"].dt.date <= pd.Timestamp("2025-03-14").date())
)

el["is_outage_period"] = outage

print(el["is_outage_period"].sum())


20862


In [ ]:
# User_ID 품질 점검

assert up["User_ID"].duplicated().sum() == 0, "User_ID 중복 존재"

orphan = set(el["User_ID"]) - set(up["User_ID"])

print("프로필 없는 User_ID 수:", len(orphan))


프로필 없는 User_ID 수: 0


In [ ]:
# category 변환 (메모리 절약)

for col in ["Event_Type", "알림_유형"]:
    if col in el.columns:
        el[col] = el[col].astype("category")

for col in ["기기", "가입경로"]:
    if col in up.columns:
        up[col] = up[col].astype("category")


In [ ]:
# 저장

up.to_csv(
    "user_profile_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

el.to_csv(
    "event_log_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

저장 완료


# 데이터 JOIN


In [ ]:
el["Event_Time"] = el["Event_Time"].dt.tz_localize(None)

print("Event_Time dtype:", el["Event_Time"].dtype)
print("가입일자 dtype   :", up["가입일자"].dtype)

Event_Time dtype: datetime64[ns]
가입일자 dtype   : datetime64[ns]


In [ ]:
# User_ID 기준 JOIN

master = pd.merge(up, el, on="User_ID", how="left")

print("master:", master.shape)
display(master.head())

master: (1757309, 12)


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,Event_Time,Event_Type,Session_ID,알림_유형,is_type_null,is_outage_period
0,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25 07:25:45,앱실행,2858201769,NaN,False,False
1,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN,False,False
2,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN,False,False
3,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25 07:27:55,챌린지참여,2858201769,NaN,False,False
4,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25 20:30:00,알림수신,NaN,광고성,False,False


In [ ]:
no_log_users = master[master["Event_Time"].isna()]["User_ID"].unique()
print("이벤트 없는 유저 수:", len(no_log_users))

no_log_df = up[up["User_ID"].isin(no_log_users)]
print("가입일자 범위:", no_log_df["가입일자"].min(), "~", no_log_df["가입일자"].max())
display(no_log_df.head())

이벤트 없는 유저 수: 47
가입일자 범위: 2025-03-10 00:00:00 ~ 2025-03-14 00:00:00


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
112,U0000113,2025-03-10,퍼포먼스광고,미확인,True,NaT
536,U0000537,2025-03-12,퍼포먼스광고,iOS,True,NaT
559,U0000560,2025-03-14,퍼포먼스광고,Android,True,NaT
871,U0000872,2025-03-10,오가닉,iOS,False,NaT
1645,U0001646,2025-03-10,오가닉,iOS,True,NaT


In [ ]:
print("[master 결측치]")
print(master.isnull().sum())

[master 결측치]
User_ID                   0
가입일자                      0
가입경로                      0
기기                        0
알림수신동의여부                  0
알림수신동의_변경일자         1609855
Event_Time               47
Event_Type            26503
Session_ID           241549
알림_유형               1538427
is_type_null             47
is_outage_period         47
dtype: int64


In [ ]:
#Day0 (온보딩 완료일) + 온보딩 완료여부

onboarding = (
    master[master["Event_Type"] == "온보딩_완료"]
    .groupby("User_ID")["Event_Time"]
    .min()
    .reset_index()
    .rename(columns={"Event_Time": "Day0_온보딩완료일"})
)

master = master.merge(onboarding, on="User_ID", how="left")
master["온보딩_완료여부"] = master["User_ID"].isin(set(onboarding["User_ID"]))

print("온보딩 완료 유저:", master["온보딩_완료여부"].groupby(master["User_ID"]).first().sum())
display(onboarding.head())


온보딩 완료 유저: 5719


,User_ID,Day0_온보딩완료일
0,U0000001,2025-01-25 07:26:15
1,U0000005,2025-02-18 12:53:07
2,U0000008,2025-02-27 15:43:25
3,U0000012,2025-04-21 14:59:29
4,U0000013,2025-01-08 07:31:10


In [ ]:
# D1 / D7 / D30 리텐션 플래그
# - Day0 = 온보딩_완료일 기준
# - is_outage_period 로그도 포함하여 계산 (팀원 원칙: 삭제 X, 결정은 분석 단계에서)


# %%
app = (
    master[
        (master["Event_Type"] == "앱실행") &
        (master["Day0_온보딩완료일"].notna())
    ]
    [["User_ID", "Event_Time", "Day0_온보딩완료일"]]
    .copy()
)

app["days_since_d0"] = (
    app["Event_Time"].dt.normalize() - app["Day0_온보딩완료일"].dt.normalize()
).dt.days

d1_users  = app[app["days_since_d0"].between(1,  2)]["User_ID"].unique()
d7_users  = app[app["days_since_d0"].between(6,  8)]["User_ID"].unique()
d30_users = app[app["days_since_d0"].between(28, 32)]["User_ID"].unique()

master["D1_retained"]  = master["User_ID"].isin(d1_users)
master["D7_retained"]  = master["User_ID"].isin(d7_users)
master["D30_retained"] = master["User_ID"].isin(d30_users)

print("D1 리텐션:", len(d1_users))
print("D7 리텐션:", len(d7_users))
print("D30 리텐션:", len(d30_users))

D1 리텐션: 4948
D7 리텐션: 3653
D30 리텐션: 2290


In [ ]:
# 첫 7일 챌린지 경험 (가입일자 기준)
ch = (
    master[master["Event_Type"] == "챌린지참여"]
    [["User_ID", "Event_Time", "가입일자"]]
    .copy()
)

ch["days_since_join"] = (
    ch["Event_Time"].dt.normalize() - ch["가입일자"].dt.normalize()
).dt.days

early_ch_users = ch[ch["days_since_join"] <= 7]["User_ID"].unique()
master["첫7일_챌린지경험"] = master["User_ID"].isin(early_ch_users)

print("첫 7일 챌린지 경험 유저:", len(early_ch_users))


첫 7일 챌린지 경험 유저: 8411


In [ ]:
# 알림동의 변경여부 + 코호트 월

master["알림동의_변경여부"] = master["알림수신동의_변경일자"].notna()
master["코호트_월"]         = master["가입일자"].dt.to_period("M").astype(str)

print(master["코호트_월"].value_counts().sort_index())

코호트_월
2025-01    284837
2025-02    772291
2025-03    282031
2025-04    248796
2025-05    169354
Name: count, dtype: int64


In [ ]:
# user_summary — 유저 1명 1행 (분석 핵심 파일)
USER_COLS = [
    "User_ID", "가입일자", "가입경로", "기기",
    "알림수신동의여부", "알림수신동의_변경일자", "알림동의_변경여부",
    "코호트_월", "Day0_온보딩완료일", "온보딩_완료여부",
    "D1_retained", "D7_retained", "D30_retained", "첫7일_챌린지경험",
]

user_summary = (
    master[USER_COLS]
    .drop_duplicates("User_ID")
    .reset_index(drop=True)
)

print("user_summary:", user_summary.shape)
display(user_summary.head())


user_summary: (12500, 14)


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,알림동의_변경여부,코호트_월,Day0_온보딩완료일,온보딩_완료여부,D1_retained,D7_retained,D30_retained,첫7일_챌린지경험
0,U0000001,2025-01-25,오가닉,iOS,True,NaT,False,2025-01,2025-01-25 07:26:15,True,True,True,True,True
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24,True,2025-05,NaT,False,False,False,False,True
2,U0000003,2025-05-14,오가닉,iOS,False,NaT,False,2025-05,NaT,False,False,False,False,False
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaT,False,2025-02,NaT,False,False,False,False,False
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaT,False,2025-02,2025-02-18 12:53:07,True,True,True,True,True


In [ ]:
# 알림 유형별 오픈율
recv = (
    master[master["Event_Type"] == "알림수신"]
    .groupby(["User_ID", "알림_유형"], observed=True).size()
    .reset_index(name="수신수")
)
open_ = (
    master[master["Event_Type"] == "알림오픈"]
    .groupby(["User_ID", "알림_유형"], observed=True).size()
    .reset_index(name="오픈수")
)

notif_summary = recv.merge(open_, on=["User_ID", "알림_유형"], how="left")
notif_summary["오픈수"] = notif_summary["오픈수"].fillna(0).astype(int)
notif_summary["오픈율"] = notif_summary["오픈수"] / notif_summary["수신수"]

print("[알림 유형별 평균 오픈율]")
print(notif_summary.groupby("알림_유형", observed=True)["오픈율"].mean().round(4))
display(notif_summary.head())


[알림 유형별 평균 오픈율]
알림_유형
광고성       0.0262
리마인드      0.1491
챌린지_알림    0.1934
Name: 오픈율, dtype: float64


,User_ID,알림_유형,수신수,오픈수,오픈율
0,U0000001,광고성,28,1,0.035714
1,U0000001,리마인드,32,3,0.093750
2,U0000001,챌린지_알림,20,3,0.150000
3,U0000004,광고성,4,0,0.000000
4,U0000004,리마인드,10,0,0.000000


In [ ]:
total = user_summary["User_ID"].nunique()
onb   = user_summary["온보딩_완료여부"].sum()
ch_n  = user_summary["첫7일_챌린지경험"].sum()
d30   = user_summary["D30_retained"].sum()

print(f"전체 유저          : {total:,}명")
print(f"온보딩 완료        : {onb:,}명 ({onb/total*100:.1f}%)")
print(f"첫7일 챌린지 경험   : {ch_n:,}명 ({ch_n/total*100:.1f}%)")
print(f"D30 리텐션         : {d30:,}명 ({d30/total*100:.1f}%)")
print()

# 주가설 미리보기
grp = user_summary.groupby("첫7일_챌린지경험")["D30_retained"].agg(["sum", "count"])
grp["rate(%)"] = (grp["sum"] / grp["count"] * 100).round(1)
print("[주가설 미리보기] 첫 7일 챌린지 경험 vs D30 리텐션")
display(grp)

전체 유저          : 12,500명
온보딩 완료        : 5,719명 (45.8%)
첫7일 챌린지 경험   : 8,411명 (67.3%)
D30 리텐션         : 2,290명 (18.3%)

[주가설 미리보기] 첫 7일 챌린지 경험 vs D30 리텐션


,sum,count,rate(%)
첫7일_챌린지경험,,,
False,151,4089,3.7
True,2139,8411,25.4


In [ ]:
master.to_csv("master.csv", index=False, encoding="utf-8-sig")
user_summary.to_csv("user_summary.csv", index=False, encoding="utf-8-sig")
notif_summary.to_csv("notification_summary.csv", index=False, encoding="utf-8-sig")

print("저장 완료:")
print("  - master.csv                 (이벤트 행 단위 전체, JOIN + 파생변수)")
print("  - user_summary.csv           (유저 1명 1행, 분석 핵심 파일)")
print("  - notification_summary.csv   (유저×알림유형 오픈율 집계)")

저장 완료:
  - master.csv                 (이벤트 행 단위 전체, JOIN + 파생변수)
  - user_summary.csv           (유저 1명 1행, 분석 핵심 파일)
  - notification_summary.csv   (유저×알림유형 오픈율 집계)


# 코호트 및 퍼널 지표 추출

# 세그먼트 심층 분석 및 시각화